In [5]:
import pandas as pd

# Load the data
df = pd.read_csv('education_data_long.csv')

# Prepare Gross Enrollment and Completion DataFrames
enrollment_df = df[df['Metric'] == 'Gross Enrollment Ratio'].pivot(index=['District', 'Year'], columns='Metric', values='Value').reset_index()
completion_df = df[df['Metric'] == 'Primary Completion Ratio'].pivot(index=['District', 'Year'], columns='Metric', values='Value').reset_index()

# Merge both on District and Year
pressure_ratio_df = pd.merge(enrollment_df, completion_df, on=['District', 'Year'], how='inner')

# Compute Enrollment Pressure Ratio safely
pressure_ratio_df['Enrollment Pressure Ratio'] = (
    pressure_ratio_df['Gross Enrollment Ratio'] /
    pressure_ratio_df['Primary Completion Ratio'].replace(0, 1e-10) * 100
).fillna(0)

# Set to 0 where Primary Completion is invalid (0 or NaN)
pressure_ratio_df['Enrollment Pressure Ratio'] = pressure_ratio_df['Enrollment Pressure Ratio'].where(
    pressure_ratio_df['Primary Completion Ratio'].notna() & (pressure_ratio_df['Primary Completion Ratio'] > 0), 0
)

# Use data for latest year only
latest_df = pressure_ratio_df[pressure_ratio_df['Year'] == pressure_ratio_df['Year'].max()]

# ✅ Compute correlation directly (no need to re-merge)
correlation = latest_df['Enrollment Pressure Ratio'].corr(latest_df['Primary Completion Ratio'])
print(f"Correlation between Enrollment Pressure Ratio and Primary Completion Ratio: {correlation:.4f}")


Correlation between Enrollment Pressure Ratio and Primary Completion Ratio: -0.6351
